# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FarisElbaz/ML_Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('hf_copllab_access')

login(token=hf_token)

print("Successfully logged in to Hugging Face Hub!")

Successfully logged in to Hugging Face Hub!


In [2]:
import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
total_rows = con.sql(f"SELECT COUNT(*) FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')")

In [3]:
rel = "hf://datasets/FlyRank/internship-warehouse"
month_path = (
    f"{rel}/fact_content_daily_performance/month=2026-03/**/*.parquet"
)

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents one piece of content on a single calender day for a specific client.
The time window is March 1st 2026 - March 31st 2026

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Context (Keys and parition boundaries):


*   client_hash_id
*   content_hash_id
*   report_date

Feauture:
*   gsc_clicks, gsc_impressions
*   gsc_avg_position
*   ga4_pageview, ga4_total_engagement_sec
*   session_organic, sessioon_ai

Label:
*   trajectory_class : Decline, Momentum, Stable/Recovery

Excluded:
*   gsc_data_available/ ga4_data_available where IS NOT TRUE: Rows lacking tracking integration allow zero-fill bias to infest data and corrupt engagment denominators
*   contet_hash_id instances with total 14-day impressions that are < 50: Excluded as unranked tail noise; percentage deltas on near-zero bases create significant artificial volatility.
*   Target window aggregates from feature matrices: Excluded to prevent temporal label leakage.












In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
q_grain = f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    COUNT(*) AS row_count
FROM read_parquet('{month_path}')
GROUP BY 1, 2, 3
HAVING COUNT(*) > 1
"""
grain_violations = con.sql(q_grain).df()
print(f"Grain Violations: {len(grain_violations)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain Violations (must be 0): 0


In [5]:
q_span = f"""
SELECT
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(*) AS total_daily_rows,
    COUNT(DISTINCT content_hash_id) AS unique_content_items,
    COUNT(DISTINCT client_hash_id) AS unique_clients
FROM read_parquet('{month_path}')
"""
print(con.sql(q_span).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  total_daily_rows  unique_content_items  \
0 2026-03-01 2026-03-31           9841378                331437   

   unique_clients  
0              55  


In [6]:
q_avail = f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS dual_available_rows,
    ROUND(COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) * 100.0 / COUNT(*), 2) AS pct_surviving
FROM read_parquet('{month_path}')
"""
print(con.sql(q_avail).df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows  dual_available_rows  \
0     9841378             3611061              413966               364347   

   pct_surviving  
0            3.7  


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Unbalanced client history depth:
Domain tenure varies across the warehouse, as such models trained across all clients simulatenously are subject to cohort imbalances, where long-tenured accounts dominate high volume trajectory representations.

GSC-Only Early Rows & Tracking Gaps:

Older panel rows or unconfigured setups lack GA4 behavioral telemetry (ga4_data_available IS FALSE), meaning bounce and dwell metrics cannot be backfilled without discarding historical records or introducing missingness bias.

Window Overlaps & Boundary Blindness: Daily aggregations do not reveal intraday spikes or real-time algorithm updates. Furthermore, a fixed 14-day observation window cuts off long-tail seasonal cycles (e.g., quarterly purchasing behavior) occurring outside the active snapshot partition.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.